# AI Agent Security submission

Run this notebook on Kaggle. It writes `/kaggle/working/attack.py`, creates a placeholder `/kaggle/working/submission.csv` for normal versioning, and starts the official inference server during competition reruns.


In [ ]:
# AI Agent Security submission - writes attack.py (base64, corruption-proof) + placeholder submission.csv
import base64, csv, os, pathlib
EXPECTED_BYTES = 19020
ATTACK_B64 = (
    '...'
    'c1swXVs6MjAwXSkK'
)
data = base64.b64decode(ATTACK_B64)
assert len(data) == EXPECTED_BYTES, f'attack.py corrupted on paste: {len(data)} != {EXPECTED_BYTES}'
src = data.decode('utf-8')
compile(src, 'attack.py', 'exec')  # fail loudly if not valid Python
out = pathlib.Path('/kaggle/working/attack.py')
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(src, encoding='utf-8')
print(f'Wrote {out} ({len(data)} bytes)  [expected {EXPECTED_BYTES}]')

# Kaggle Submit checks the committed version outputs submission.csv; the official
# rerun overwrites it with real scores. These zeros are just a valid placeholder.
with open('/kaggle/working/submission.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['Id', 'Score'])
    for rid in ['gpt_oss_public', 'gpt_oss_private', 'gemma_public', 'gemma_private']:
        w.writerow([rid, 0])
print('Wrote placeholder /kaggle/working/submission.csv (overwritten by the official rerun)')

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    from kaggle_evaluation.jed_attack_134815.jed_attack_inference_server import (
        JEDAttackInferenceServer,
    )

    JEDAttackInferenceServer().run()
else:
    print('Not a competition rerun; server startup skipped for normal notebook save/run.')
